# Day02 · 작은 계산에서 전체 학습 흐름까지

`Day01 역할 회수 → 모델로 묶기 → 패턴 조사 → 분류 계약 → 학습 관리 → 데이터 묶음 → 평가 → 진단과 다음 질문`

- 결론을 먼저 외우지 않고 실행 전 예상과 관찰을 연결함
- 준비된 셀을 실행하고 값·shape·dtype·device를 함께 확인함
- 실습 답은 별도 Practice A에서만 확인함

## 오늘 확인할 것

1. Day1 계산 책임을 다른 변수명에서 다시 찾음
2. 직접 관리하던 값을 모델이 모아 관리하는 방식과 비교함
3. 행렬곱과 `nn.Linear`가 같은 계산을 만드는지 확인함
4. 네 입력에 대한 판정을 먼저 예상하고 실행 결과로 확인함
5. class별 원점수와 loss의 입력 계약을 연결함
6. optimizer, 현재 batch, device, 평가 흐름을 한 pipeline으로 묶음
7. shape·dtype·device를 기준으로 오류를 진단하고 다음 구조의 필요를 질문함

각 실습은 같은 PC 번호의 Practice Q에서 먼저 시도하고 Practice A에서 답을 확인함.

In [104]:
import copy
import warnings

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.set_printoptions(precision=6, sci_mode=False)
print("준비 완")

준비 완


## Day1 핵심 연산 회수

Day1 내용을 다시 길게 배우지 않음. 작은 새 값에서 질문에 맞는 연산을 고르고 결과 shape을 먼저 예측함.

- 값을 만들고 모양을 바꾸는가(`create`/`reshape`), 일부를 고르는가(indexing/slicing)?
- 크기 1인 축을 없애거나 더하는가(`squeeze`/`unsqueeze`), Tensor를 기존 축 또는 새 축으로 합치는가(`cat` vs `stack`)?
- 같은 위치끼리 곱하는가(`*`), 안쪽 축을 연결해 행렬곱하는가(`@`/`matmul`)?

연산 이름부터 외우지 않고 질문을 고른 뒤, 결과 shape을 예상하고 준비된 셀을 실행함.

In [105]:
retrieval_grid = torch.arange(15, dtype=torch.float32).reshape(3, 5)
retrieval_row = retrieval_grid[1]
retrieval_slice = retrieval_grid[:2, 1:4]

retrieval_with_axis = torch.arange(6, dtype=torch.float32).reshape(3, 1, 2)
retrieval_squeezed = retrieval_with_axis.squeeze(1)
retrieval_column = retrieval_squeezed.unsqueeze(-1)

retrieval_left = torch.tensor([[1., 2.], [3., 4.]])
retrieval_right = torch.tensor([[10., 20.], [30., 40.]])
retrieval_cat = torch.cat([retrieval_left, retrieval_right], dim=0)
retrieval_stack = torch.stack([retrieval_left, retrieval_right], dim=1)
retrieval_elementwise = retrieval_left * retrieval_right
retrieval_matmul = retrieval_left @ retrieval_right

print("create / reshape:", retrieval_grid.shape)
print("index / slice:", retrieval_row.shape, retrieval_slice.shape)
print("shape:", retrieval_squeezed.shape, retrieval_column.shape)
print("cat / stack:", retrieval_cat.shape, retrieval_stack.shape)
print("* / matmul:", retrieval_elementwise.shape, retrieval_matmul.shape)

create / reshape: torch.Size([3, 5])
index / slice: torch.Size([5]) torch.Size([2, 3])
shape: torch.Size([3, 2]) torch.Size([3, 2, 1])
cat / stack: torch.Size([4, 2]) torch.Size([2, 2, 2])
* / matmul: torch.Size([2, 2]) torch.Size([2, 2])


### 핵심 연산 관찰

- Tensor를 만든 뒤 `reshape`는 원소 수를 유지하며 모양을 바꿈. indexing/slicing은 필요한 위치나 구간을 고름.
- `squeeze(축)`는 크기 1인 축을 없애고 `unsqueeze(축)`는 크기 1인 축을 새로 만듦.
- `cat`은 기존 축을 이어 붙이고 `stack`은 새 축을 만듦. `*`는 같은 위치의 원소끼리 곱하고 `@`/`matmul`은 안쪽 축을 연결함.

Practice에서는 예제 숫자를 바꾸고 질문과 예상 shape을 근거로 연산을 직접 선택함.

## 평가 대비 짧은 API 확인

여기는 새 이론 수업이 아님. 이미 본 계산을 짧게 회수하고 API 이름과 결과를 알아보는 구간임.

In [106]:
retrieval_decimal = torch.tensor([3.7, -2.4, 0.9])
retrieval_long = retrieval_decimal.long()
retrieval_float = retrieval_long.float()

retrieval_sigmoid = torch.sigmoid(torch.tensor([-2.0, 0.0, 2.0]))
retrieval_prediction = torch.tensor([1.0, 3.0, 5.0])
retrieval_target = torch.tensor([2.0, 1.0, 5.0])
retrieval_mse_manual = ((retrieval_prediction - retrieval_target) ** 2).mean()
retrieval_mse_builtin = nn.MSELoss()(retrieval_prediction, retrieval_target)

print("dtype:", retrieval_decimal.dtype, retrieval_long.dtype, retrieval_float.dtype)
print("cast values:", retrieval_decimal, "->", retrieval_long, "->", retrieval_float)
print("sigmoid center:", retrieval_sigmoid[1].item())
print("MSE manual / nn.MSELoss:", retrieval_mse_manual.item(), retrieval_mse_builtin.item())

dtype: torch.float32 torch.int64 torch.float32
cast values: tensor([ 3.700000, -2.400000,  0.900000]) -> tensor([ 3, -2,  0]) -> tensor([ 3., -2.,  0.])
sigmoid center: 0.5
MSE manual / nn.MSELoss: 1.6666666269302368 1.6666666269302368


### 짧게 확인할 경계

- `.long()`은 정수 dtype으로 바꾸며 소수부를 버림. 반올림이 아니고, 뒤에서 `.float()`을 적용해도 이미 버린 소수부는 돌아오지 않음.
- `Sigmoid`에 입력 `0`을 넣으면 출력은 `0.5`임. 여기서는 적용과 결과 인식만 하며 새 이진 분류 이론으로 확장하지 않음.
- 직접 계산한 오차 제곱 평균과 `nn.MSELoss()`의 기본 mean 결과가 같음. 새 이론이 아니라 같은 MSE 계산을 API로 확인한 것임.

## Day1 역할 찾기

아래 코드에서 먼저 표시할 역할

1. input과 target
2. 학습으로 바뀌는 parameter
3. prediction과 loss
4. gradient 계산
5. parameter update
6. gradient reset

In [107]:
feature = torch.tensor([2.0])
answer = torch.tensor([5.0])
scale = torch.tensor([1.0], requires_grad=True)
offset = torch.tensor([0.0], requires_grad=True)

estimate = scale * feature + offset
distance = ((estimate - answer) ** 2).mean()
distance.backward()

before = (scale.item(), offset.item(), distance.item())
with torch.no_grad():
    scale -= 0.05 * scale.grad
    offset -= 0.05 * offset.grad
    # w -= 0.05 * scale.grad
    # b -= 0.05 * offset.grad
scale.grad.zero_()
offset.grad.zero_()

fresh_estimate = scale * feature + offset
fresh_distance = ((fresh_estimate - answer) ** 2).mean()
after = (scale.item(), offset.item(), fresh_distance.item())

print("before (scale, offset, loss):", before)
print("after  (scale, offset, loss):", after)
print("reset grads:", scale.grad.item(), offset.grad.item())

before (scale, offset, loss): (1.0, 0.0, 9.0)
after  (scale, offset, loss): (1.600000023841858, 0.30000001192092896, 2.25)
reset grads: 0.0 0.0


## `b`를 다시 움직이는 값으로 붙이기

Day1의 local-slope 관찰에서는 `b`를 고정하고 `w`만 움직였음.  
이제 `w`와 `b`를 모두 학습 대상으로 두고 같은 책임을 각각 수행함.

In [108]:
bias_w = torch.tensor([1.0], requires_grad=True)
bias_b = torch.tensor([0.0], requires_grad=True)
bias_x = torch.tensor([2.0])
bias_target = torch.tensor([5.0])

bias_prediction = bias_w * bias_x + bias_b
bias_loss = ((bias_prediction - bias_target) ** 2).mean()
bias_loss.backward()
print("grad 확인 × 2:", bias_w.grad.item(), bias_b.grad.item())

with torch.no_grad():
    bias_w -= 0.05 * bias_w.grad
    bias_b -= 0.05 * bias_b.grad
print("update × 2:", bias_w.item(), bias_b.item())
print("update 뒤에도 grad 유지:", bias_w.grad.item(), bias_b.grad.item())

bias_fresh_prediction = bias_w * bias_x + bias_b
bias_fresh_loss = ((bias_fresh_prediction - bias_target) ** 2).mean()
print("fresh prediction / loss:", bias_fresh_prediction.item(), bias_fresh_loss.item())

bias_w.grad.zero_()
bias_b.grad.zero_()
print("reset × 2:", bias_w.grad.item(), bias_b.grad.item())

grad 확인 × 2: -12.0 -6.0
update × 2: 1.600000023841858 0.30000001192092896
update 뒤에도 grad 유지: -12.0 -6.0
fresh prediction / loss: 3.5 2.25
reset × 2: 0.0 0.0


### 보이는 관리 비용

| 책임 | `w` | `b` |
|---|---|---|
| gradient 확인 | 따로 확인 | 따로 확인 |
| update | 따로 작성 | 따로 작성 |
| reset | 따로 작성 | 따로 작성 |

두 개는 가능함. 학습할 값이 더 많아지면 같은 책임을 모아 다룰 구조가 필요함.

### 다음 필요

학습할 값을 **모아 찾고**, 입력에서 출력까지의 계산을 **한 모델 안에 묶는 방법**을 찾음.

### Practice PC-01

이번 Practice에서 확인할 것: 수동 update의 gradient 계산·새 결과 확인·reset 책임을 구분함.

`DAY02_PRACTICE_Q.ipynb`의 **PC-01**로 이동함.  
답을 확인한 뒤 **직접 관리에서 모델 관리로** 제목으로 돌아옴.

## 직접 관리에서 모델 관리로

- 모델: input과 현재 내부 숫자(parameter)를 사용해 output을 만드는 계산 규칙
- parameter: 모델 내부의 조절 가능한 값; parameter라는 이름이 정답을 뜻하지 않음
- `forward`: 현재 input과 parameter로 output을 계산함; update가 아님
- `nn.Module`: parameter 상태와 forward 구조를 정리해 담는 그릇; 자동 학습 장치가 아님

![모델의 현재 계산과 별도 업데이트 책임](assets/day02/model_forward_parameter_dark.svg)

In [119]:
class ManagedAffine(nn.Module):
    def __init__(self, start_w=1.6, start_b=0.3):
        super().__init__()
        self.w = nn.Parameter(torch.tensor([start_w]))
        self.b = nn.Parameter(torch.tensor([start_b]))

    def forward(self, value):
        return self.w * value + self.b

managed_affine = ManagedAffine()
print("registered parameters:")
for name, parameter in managed_affine.named_parameters():
    print(name, type(parameter).__name__, tuple(parameter.shape), parameter.item())
print("forward result:", managed_affine(torch.tensor([2.0])).item())
torch.tensor([2.0]).item()
print(list(managed_affine.parameters()))

registered parameters:
w Parameter (1,) 1.600000023841858
b Parameter (1,) 0.30000001192092896
forward result: 3.5
[Parameter containing:
tensor([1.600000], requires_grad=True), Parameter containing:
tensor([0.300000], requires_grad=True)]


### 관찰

- `nn.Parameter`: Module 속성에 놓이면 학습 대상 parameter로 등록됨
- `nn.Module`: parameter 상태와 `forward()` 계산을 함께 관리함
- `named_parameters()`: 등록된 이름과 값을 모아 확인함
- **등록됨 ≠ 자동 업데이트됨**

### 자주 쓰는 `입력에 weight를 곱하고 bias를 더하는 계산`

입력 feature 1개로 output 1개를 만드는 계산을 매번 직접 쓰지 않고 `nn.Linear`로 묶어 사용할 수 있음.

### Day1 행렬곱과 `nn.Linear` 연결하기

- Day1 직접 계산: `X [batch,in] @ W [in,out] -> [batch,out]`
- `nn.Linear` 저장: `weight [out,in]`
- 같은 forward 계산: `X @ weight + bias`

`X`의 각 행은 한 입력이고, `weight`를 사용하면 Day1의 `[in,out]` 방향과 맞음. PyTorch가 왜 weight를 이 방향으로 저장하는지는 외울 필요 없음. 이번 목표는 Day1 직접 행렬곱과 같은 output이 나오는지만 확인하는 것임.

In [120]:
orientation_x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
orientation_linear = nn.Linear(2, 2)
with torch.no_grad():
    orientation_linear.weight.copy_(torch.tensor([[0.5, -1.0], [1.5, 0.25]]))
    orientation_linear.bias.copy_(torch.tensor([0.1, -0.2]))

orientation_from_linear = orientation_linear(orientation_x)
orientation_manual = orientation_x @ orientation_linear.weight.T + orientation_linear.bias

print("X shape:", tuple(orientation_x.shape))
print("stored weight shape [out,in]:", tuple(orientation_linear.weight.shape))
print("nn.Linear output:", orientation_from_linear)
print("manual output:", orientation_manual)
print("same output:", torch.allclose(orientation_from_linear, orientation_manual))
print("max difference:", (orientation_from_linear - orientation_manual).abs().max().item())

X shape: (2, 2)
stored weight shape [out,in]: (2, 2)
nn.Linear output: tensor([[-1.400000,  1.800000],
        [-2.400000,  5.300000]], grad_fn=<AddmmBackward0>)
manual output: tensor([[-1.400000,  1.800000],
        [-2.400000,  5.300000]], grad_fn=<AddBackward0>)
same output: True
max difference: 0.0


# 02-3 Linear를 모델 안에 넣기

In [124]:
class OneLinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, value):
        return self.linear(value)

one_linear_model = OneLinearModel()
with torch.no_grad():
    one_linear_model.linear.weight.copy_(torch.tensor([[1.6]]))
    one_linear_model.linear.bias.copy_(torch.tensor([0.3]))

linear_input = torch.tensor([[2.0]])
linear_target = torch.tensor([[5.0]])
linear_before = one_linear_model(linear_input)
linear_loss = ((linear_before - linear_target) ** 2).mean()
linear_loss.backward()
with torch.no_grad():
    for parameter in one_linear_model.parameters():
        parameter -= 0.05 * parameter.grad
        parameter.grad.zero_()
linear_fresh = one_linear_model(linear_input)
linear_fresh_loss = ((linear_fresh - linear_target) ** 2).mean()

one_linear_model = OneLinearModel()
print("registered names:", [name for name, _ in one_linear_model.named_parameters()])
print("weight shape:", tuple(one_linear_model.linear.weight.shape))
print("bias shape:", tuple(one_linear_model.linear.bias.shape))
print("before prediction / loss:", linear_before.item(), linear_loss.item())
print("fresh prediction / loss:", linear_fresh.item(), linear_fresh_loss.item())
print("reset grads:", [parameter.grad.item() for parameter in one_linear_model.parameters()])

registered names: ['linear.weight', 'linear.bias']
weight shape: (1, 1)
bias shape: (1,)
before prediction / loss: 3.5 2.25
fresh prediction / loss: 4.25 0.5625
reset grads: [0.0, 0.0]


### 한 흐름으로 회수

`직접 Tensor 관리 → Parameter 등록 → Module로 상태·계산 묶기 → Linear로 곱하고 더하는 계산 재사용`

class 안의 `Linear`도 `parameters()`로 모아 같은 update/reset을 수행하며, 바뀐 값으로 fresh output을 다시 확인함.

### Practice PC-02

이번 Practice에서 확인할 것: 같은 계산을 등록된 parameter와 `forward` 구조로 옮김.

Practice Q의 **PC-02**로 이동함.  
답을 확인한 뒤 **하나의 Linear 경계** 제목으로 돌아옴.

## 하나의 Linear 경계

1. 입력 하나에는 두 feature `x1`, `x2`가 있음.
2. `x1*w1 + x2*w2 + b`를 한 번 계산하면 숫자 score 하나가 나옴.
3. `score >= 0` 같은 간단한 기준을 적용하면 0/1 결정을 만들 수 있음.
4. 이 2차원 예제에서 이런 규칙 하나는 점을 나누는 직선 경계 하나에 해당함.
5. 입력 shape `[4,2]`는 **예제 4개 × 각 예제의 feature 2개**라는 뜻임.

![두 입력 Linear unit이 score를 만드는 흐름](assets/day02/linear_unit_score_dark.svg)

AND는 둘 다 1일 때만 1, OR은 하나라도 1이면 1임. **XOR은 둘 중 정확히 하나만 1일 때 1**임.

### 관찰 뒤 정확한 용어

- 위 계산 단위를 **linear unit**이라고 부름.
- `Linear(2,1)`은 feature 2개를 받아 output score 1개를 만듦.
- `out_features`는 output unit, 즉 linear combination의 개수임. `nn.Linear`가 항상 한 neuron이라는 뜻은 아님.
- **score ≠ class label**: score는 숫자 계산 결과이고, class label 0/1은 기준을 적용해 읽은 결정임.
- 직선 하나로 두 무리를 나눌 수 있는지를 **선형 분리 가능성(linear separability)**이라고 부름.
- MSE target shape `[4,1]`은 같은 네 예제에 output 하나씩 대응함.

In [125]:
logic_x = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])

def fixed_boundary(weight, bias):
    scores = logic_x @ torch.tensor(weight, dtype=torch.float32).reshape(2, 1) + bias
    decisions = (scores >= 0).to(torch.int64).squeeze(1)
    return scores.squeeze(1), decisions

and_scores, and_decisions = fixed_boundary([1., 1.], -1.5)
or_scores, or_decisions = fixed_boundary([1., 1.], -0.5)
print("inputs:", logic_x.tolist())
print("AND scores / decisions:", and_scores.tolist(), and_decisions.tolist())
print("OR  scores / decisions:", or_scores.tolist(), or_decisions.tolist())

inputs: [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
AND scores / decisions: [-1.5, -0.5, -0.5, 0.5] [0, 0, 0, 1]
OR  scores / decisions: [-0.5, 0.5, 0.5, 1.5] [0, 1, 1, 1]


### Prediction 1 · 실행 전에 기록

**`XOR도 weight/bias만 잘 고르면 되지 않을까?`**

AND와 OR이 성공한 근거를 사용해 한 문장으로 예상함.

In [126]:
xor_target = torch.tensor([0, 1, 1, 0])
xor_trials = [
    ("후보 A", [1., 1.], -0.5),
    ("후보 B", [1., 1.], -1.5),
    ("후보 C", [1., -1.], -0.5),
]
for label, weight, bias in xor_trials:
    scores, decisions = fixed_boundary(weight, bias)
    accuracy = (decisions == xor_target).float().mean().item()
    print(label, "decisions=", decisions.tolist(), "accuracy=", accuracy)

후보 A decisions= [0, 1, 1, 1] accuracy= 0.75
후보 B decisions= [0, 0, 0, 1] accuracy= 0.25
후보 C decisions= [0, 0, 1, 0] accuracy= 0.75


# 기하학

![AND OR XOR 선형 경계 비교](assets/day02/linear_boundary_dark.svg)

In [127]:
torch.manual_seed(7)
single_linear = nn.Linear(2, 1)
xor_float_target = xor_target.to(torch.float32).reshape(4, 1)
single_history = []

for step in range(500):
    single_output = single_linear(logic_x)
    single_loss = ((single_output - xor_float_target) ** 2).mean()
    if step in (0, 9, 49, 99, 499):
        single_history.append((step + 1, single_loss.detach().item()))
    single_loss.backward()
    with torch.no_grad():
        for parameter in single_linear.parameters():
            parameter -= 0.1 * parameter.grad
            parameter.grad.zero_()

with torch.no_grad():
    single_output = single_linear(logic_x)
    single_loss = ((single_output - xor_float_target) ** 2).mean()
    single_decisions = (single_output >= 0.5).to(torch.int64).squeeze(1)

print("loss history:", [(s, round(v, 6)) for s, v in single_history])
print("final outputs:", [round(v, 6) for v in single_output.squeeze(1).tolist()])
print("final decisions:", single_decisions.tolist())
print("final MSE:", round(single_loss.item(), 6))

loss history: [(1, 0.510454), (10, 0.261518), (50, 0.25019), (100, 0.250001), (500, 0.25)]
final outputs: [0.5, 0.5, 0.5, 0.5]
final decisions: [1, 1, 1, 0]
final MSE: 0.25


### 실행 실패와 구조적 한계를 구분함

- 준비된 세 후보가 모두 한 점 이상 틀림
- 학습된 한 Linear도 네 점을 모두 맞히지 못함
- 그러나 학습 plateau만으로 표현 불가능을 증명하지 않음
- XOR의 1 두 점은 대각선, 0 두 점은 반대 대각선에 놓임
- 어떤 한 직선도 두 클래스 사이를 가르지 못함

결론: **하나의 선형 결정 경계로 XOR을 표현할 수 없음**

### Practice PC-03

이번 Practice에서 확인할 것: AND/OR와 XOR를 직선 하나로 나눌 수 있는지 비교함.

Practice Q의 **PC-03**으로 이동함.  
답을 확인한 뒤 **층을 여러 개 쌓으면?** 제목으로 돌아옴.

## 층을 여러 개 쌓으면?

- layer: 여러 unit이 intermediate value의 vector를 만드는 계산 묶음
- hidden layer: input feature와 final output 사이의 intermediate representation layer
- hidden은 비밀이라는 뜻이 아니라 직접 감독되는 최종 output이 아니라는 뜻
- hidden width `16`: intermediate unit/value 16개, 즉 **hidden units = 16**

![입력에서 hidden units를 거쳐 output으로 가는 구조](assets/day02/hidden_layer_structure_dark.svg)

층을 더 놓았을 때 전체 계산이 어떻게 달라지는지 실행 전에 먼저 예상함.

### Prediction 2 · 실행 전에 기록

**`Linear를 여러 층 쌓기만 하면 XOR가 해결될까?`**

층 수와 경계 모양의 관계를 한 문장으로 예상함.

In [130]:
torch.manual_seed(11)
linear_1 = nn.Linear(2, 4)
linear_2 = nn.Linear(4, 1)

stacked_output = linear_2(linear_1(logic_x))
combined_weight = linear_2.weight @ linear_1.weight
combined_bias = linear_2.weight @ linear_1.bias + linear_2.bias
combined_output = logic_x @ combined_weight.T + combined_bias

print("stacked shape:", tuple(stacked_output.shape))
print("combined shape:", tuple(combined_output.shape))
print("max difference:", (stacked_output - combined_output).abs().max().item())

stacked shape: (4, 1)
combined shape: (4, 1)
max difference: 0.0


![비선형성이 필요한 이유](assets/day02/nonlinearity_need_dark.svg)

In [129]:
relu_demo = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
print("before ReLU:", relu_demo.tolist())
print("after  ReLU:", torch.relu(relu_demo).tolist())

before ReLU: [-2.0, -0.5, 0.0, 0.5, 2.0]
after  ReLU: [0.0, 0.0, 0.0, 0.5, 2.0]


### 첫 XOR MLP 계약

- 구조: `Linear(2,16) → ReLU → Linear(16,1)`
- target: float `[4,1]`
- loss: 이미 배운 MSE
- update/reset: parameter 목록을 순회해 직접 수행
- 실행 조건: hidden width 16, learning rate 0.1, 500 step

In [131]:
class XORMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 16)
        self.output = nn.Linear(16, 1)

    def forward(self, value):
        hidden_value = torch.relu(self.hidden(value))
        return self.output(hidden_value)

torch.manual_seed(42)
xor_mlp = XORMLP()
xor_history = []

for step in range(500):
    xor_output = xor_mlp(logic_x)
    xor_loss = ((xor_output - xor_float_target) ** 2).mean()
    if step in (0, 9, 49, 99, 199, 299, 499):
        xor_history.append((step + 1, xor_loss.detach().item()))
    xor_loss.backward()
    with torch.no_grad():
        for parameter in xor_mlp.parameters():
            parameter -= 0.1 * parameter.grad
            parameter.grad.zero_()

with torch.no_grad():
    xor_output = xor_mlp(logic_x)
    xor_loss = ((xor_output - xor_float_target) ** 2).mean()
    xor_decisions = (xor_output >= 0.5).to(torch.int64).squeeze(1)

print("loss history:", [(s, round(v, 6)) for s, v in xor_history])
print("final outputs:", [round(v, 6) for v in xor_output.squeeze(1).tolist()])
print("final decisions:", xor_decisions.tolist())
print("final MSE:", f"{xor_loss.item():.10f}")

loss history: [(1, 0.373959), (10, 0.263891), (50, 0.145432), (100, 0.027966), (200, 9.1e-05), (300, 0.0), (500, 0.0)]
final outputs: [1e-06, 0.999999, 0.999999, 1e-06]
final decisions: [0, 1, 1, 0]
final MSE: 0.0000000000


![XOR MLP 학습 증거](assets/day02/xor_mlp_evidence_dark.svg)

### 관찰과 경계

- Linear 두 층만 연결한 출력은 하나의 Linear와 같음
- activation/nonlinearity: 전체가 하나의 affine rule로 다시 합쳐지지 않게 하는 변환 역할
- ReLU: 음수는 0, 0과 양수는 그대로 두어 중간 표현을 꺾음
- 같은 네 점에서 한 Linear는 실패하고 MLP는 XOR 판정을 모두 맞힘
- MLP(다층 퍼셉트론): 하나 이상의 hidden layer와 nonlinear activation을 연결한 구조 이름
- MLP는 특별한 PyTorch API나 필수 class 이름이 아님
- 낮은 training MSE는 이 네 점의 성공 근거임
- 일반화 성능이나 모든 초기화의 성공을 뜻하지 않음
- 분류 출력·loss 계약은 다음 학습 범위에서 다룸

### Practice PC-04

이번 Practice에서 확인할 것: 두 Linear 사이 ReLU 위치와 XOR 학습 구조를 확인함.

Practice Q의 **PC-04**로 이동함.  
답을 확인한 뒤 **분류 모델의 출력과 loss 계약** 제목으로 돌아옴.

## 앞부분 연결

`직접 w,b 관리 → 등록된 model state → 하나의 Linear 경계 → XOR 반례 → hidden layer + ReLU → XOR MLP`

- XOR은 두 입력 중 정확히 하나만 1일 때 1임
- 한 Linear는 XOR을 분리하지 못함
- hidden layer와 ReLU를 넣은 MLP는 같은 네 점을 구분함
- 이 MSE 예시는 비선형성이 필요한 이유를 보기 위한 계약이며, 이제 일반 분류의 출력과 loss 계약을 새로 확인함

## 분류 모델의 출력과 loss 계약

XOR MSE 장면에서는 output 하나를 0 또는 1에 가깝게 맞췄음. 일반적인 두-class 분류에서는 한 sample마다 **class 0 점수와 class 1 점수** 두 개를 먼저 낼 수 있음. 확률은 아직 필요하지 않음.

In [ ]:
example_logits = torch.tensor([[2.0, 0.5], [-0.2, 1.4], [0.1, 0.0]])
example_targets = torch.tensor([0, 1, 0], dtype=torch.int64)
ce = nn.CrossEntropyLoss()
example_loss = ce(example_logits, example_targets)
print("class별 raw scores shape:", tuple(example_logits.shape))
print("target shape / dtype:", tuple(example_targets.shape), example_targets.dtype)
print("CrossEntropyLoss:", round(example_loss.item(), 6))
print("predicted classes:", example_logits.argmax(dim=1).tolist())

### 관찰 뒤 정확한 용어

- 한 sample의 class별 **원점수**를 `logits`라고 부름
- batch logits shape는 `[N, 2]`, 정답은 sample마다 class 번호 하나이므로 `[N]`의 integer Tensor임
- 현재 수업 계약: `CrossEntropyLoss(raw logits, integer class target)`
- logits는 확률이 아니며 합이 1일 필요가 없음

### Prediction 3 · 실행 전에 기록

**분류 결과를 확률처럼 보고 싶다면, Softmax를 먼저 적용한 값을 CrossEntropyLoss에 넣어도 같은 학습 계약일까?**

“실행되는가?”와 “같은 의미의 학습인가?”를 나누어 예상함.

In [ ]:
raw_for_compare = torch.tensor(
    [[2.0, 0.5, -1.0], [-0.2, 1.4, 0.3], [0.0, 0.0, 0.0]],
    requires_grad=True,
)
target_for_compare = torch.tensor([0, 2, 1])
direct_ce = nn.functional.cross_entropy(raw_for_compare, target_for_compare)
direct_grad = torch.autograd.grad(direct_ce, raw_for_compare)[0]

raw_for_softmax = raw_for_compare.detach().clone().requires_grad_(True)
probabilities = torch.softmax(raw_for_softmax, dim=1)
softmax_then_ce = nn.functional.cross_entropy(probabilities, target_for_compare)
softmax_grad = torch.autograd.grad(softmax_then_ce, raw_for_softmax)[0]

print("두 경로 모두 실행:", True)
print("raw logits → CE:", round(direct_ce.item(), 6))
print("Softmax → CE:", round(softmax_then_ce.item(), 6))
print("loss가 같음:", torch.allclose(direct_ce, softmax_then_ce))
print("gradient가 같음:", torch.allclose(direct_grad, softmax_grad))
print("Softmax row sums:", probabilities.sum(dim=1).tolist())

![raw logits와 CrossEntropyLoss 계약](assets/day02/softmax_crossentropy_trap_dark.svg)

Softmax-before-CE도 이 실행에서는 오류 없이 동작했지만 loss와 gradient가 달랐음. 따라서 “실행 불가”가 아니라 **현재 채택한 학습 계약과 다름**이라고 말함. Softmax는 학습 뒤 logits를 확률 비율처럼 읽는 단계에서 사용할 수 있음.

그 이유는 `CrossEntropyLoss`가 내부에서 class별 점수를 `log_softmax`로 바꾼 뒤 정답 class의 값을 읽는 계산을 이미 포함하기 때문임. 앞에서 Softmax를 한 번 더 적용하면 같은 계약을 두 번 거치게 되어 loss와 gradient가 달라짐.

### output 두 개가 필요한 MLP

입력 두 값을 중간 값 16개로 바꾸고, 마지막에는 class 0과 class 1의 원점수 두 개를 냄. 이 필요를 코드로 옮기면 `2 → 16 → 2`가 됨.

In [ ]:
class ClassificationMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 16)
        self.output = nn.Linear(16, 2)

    def forward(self, value):
        hidden_value = torch.relu(self.hidden(value))
        return self.output(hidden_value)  # raw logits

classification_model = ClassificationMLP()
parameter_rows = []
for name, parameter in classification_model.named_parameters():
    parameter_rows.append((name, tuple(parameter.shape), parameter.numel()))
total_parameters = sum(row[2] for row in parameter_rows)
for row in parameter_rows:
    print(row)
print("total trainable parameters:", total_parameters)
assert total_parameters == 82

![2에서 16을 거쳐 2로 가는 MLP의 parameter 수](assets/day02/mlp_2_16_2_param_count_dark.svg)

첫 Linear는 `32 weight + 16 bias = 48`, 둘째 Linear는 `32 weight + 2 bias = 34`; 실제 named parameter shape에서 합계 **82**를 확인함. ReLU에는 이 모델에서 학습 parameter가 없음.

### 코드 읽기 확장 · `nn.Sequential`

같은 `Linear → ReLU → Linear` 순서를 `nn.Sequential(...)`로 묶은 코드를 볼 수도 있음. 이번 핵심은 class 작성법과 Sequential 중 하나를 외우는 것이 아니라, **두 Linear 사이의 ReLU와 마지막 raw logits 두 개**를 찾는 것임.

### Practice PC-05

이번 Practice에서 확인할 것: raw logits·target 계약과 `2→16→2` parameter 수를 확인함.

Practice Q의 **PC-05**로 이동함.  
답을 확인한 뒤 **optimizer가 맡는 책임** 제목으로 돌아옴.

## optimizer가 맡는 책임

parameter가 82개라면 각 값을 이름으로 찾아 직접 update하기보다, 등록된 parameter 목록을 optimizer에 맡길 수 있음. 그러나 prediction, loss, gradient 계산까지 사라지는 것은 아님.

### Prediction 4 · 실행 전에 기록

**`optimizer.step()`으로 값을 바꾼 뒤 gradient도 자동으로 사라질까?**

In [ ]:
grad_probe = nn.Linear(1, 1, bias=False)
with torch.no_grad():
    grad_probe.weight.fill_(1.0)
grad_optimizer = torch.optim.SGD(grad_probe.parameters(), lr=0.1)
grad_optimizer.zero_grad(set_to_none=True)
grad_loss = ((grad_probe(torch.tensor([[2.0]])) - torch.tensor([[0.0]])) ** 2).mean()
grad_loss.backward()
grad_before_step = grad_probe.weight.grad.detach().clone()
weight_before_step = grad_probe.weight.detach().clone()
grad_optimizer.step()
grad_after_step = grad_probe.weight.grad.detach().clone()
weight_after_step = grad_probe.weight.detach().clone()
grad_optimizer.zero_grad(set_to_none=True)
print("weight before / after step:", weight_before_step.item(), weight_after_step.item())
print("grad before / after step:", grad_before_step.item(), grad_after_step.item())
print("grad after zero_grad(set_to_none=True):", grad_probe.weight.grad)

### 책임 순서

`지난 gradient 정리 → 현재 값으로 예측 → 오차 계산 → gradient 계산 → 값 갱신`

정확한 API 순서는 `zero_grad → forward → loss → backward → step`임. `backward()`가 gradient를 만들고, `step()`은 그 gradient로 parameter 값을 바꾸며, `zero_grad()`가 이전 gradient를 정리함.

![한 batch를 학습하는 기본 순서](assets/day02/canonical_training_loop_dark.svg)

### 네 점에서 더 많은 점으로

XOR 네 점으로 구조의 필요를 확인했음. 이제 optimizer와 batch 흐름을 반복해서 관찰하기 위해, 서로 휘어진 두 무리의 점 500개를 준비함. 정답을 맞히는 새 이론이 아니라 **같은 학습 책임을 여러 sample에 반복하는 연습용 데이터**임.

In [ ]:
def make_two_moons(n=500, noise=0.08, seed=20260903):
    half = n // 2
    angle = torch.linspace(0, torch.pi, half)
    first = torch.stack([torch.cos(angle), torch.sin(angle)], dim=1)
    second = torch.stack([1.0 - torch.cos(angle), 0.45 - torch.sin(angle)], dim=1)
    features = torch.cat([first, second], dim=0)
    targets = torch.cat([
        torch.zeros(half, dtype=torch.int64),
        torch.ones(n - half, dtype=torch.int64),
    ])
    generator = torch.Generator().manual_seed(seed)
    features = features + noise * torch.randn(features.shape, generator=generator)
    return features.to(torch.float32), targets

day02_features, day02_targets = make_two_moons()
torch.manual_seed(20260903)
loop_model = ClassificationMLP()
loop_optimizer = torch.optim.Adam(loop_model.parameters(), lr=0.03)
loop_criterion = nn.CrossEntropyLoss()
loop_losses = []
for step in range(80):
    loop_optimizer.zero_grad()
    loop_logits = loop_model(day02_features)
    loop_loss = loop_criterion(loop_logits, day02_targets)
    loop_loss.backward()
    loop_optimizer.step()
    if step in (0, 9, 39, 79):
        loop_losses.append((step + 1, round(loop_loss.item(), 6)))
with torch.no_grad():
    loop_accuracy = (loop_model(day02_features).argmax(dim=1) == day02_targets).float().mean().item()
print("loss checkpoints:", loop_losses)
print("training accuracy:", round(loop_accuracy, 4))

### learning rate와 optimizer를 함께 읽기

Learning rate는 gradient가 가리킨 방향으로 한 번에 얼마나 움직일지 정하는 크기임. 너무 큰 값이 언제나 빠르고 좋은 것은 아님.

먼저 값 하나를 3에 가깝게 움직이는 작은 문제에서 크기 차이를 직접 확인함.

### Prediction 5

**같은 데이터와 출발점을 사용하면 Adam이 SGD보다 항상 나을까?**

In [ ]:
lr_probe_summary = {}
for probe_lr in (0.1, 0.5, 1.1):
    probe_value = nn.Parameter(torch.tensor([0.0]))
    probe_optimizer = torch.optim.SGD([probe_value], lr=probe_lr)
    for _ in range(12):
        probe_optimizer.zero_grad()
        probe_loss = ((probe_value - 3.0) ** 2).sum()
        probe_loss.backward()
        probe_optimizer.step()
    final_probe_loss = ((probe_value - 3.0) ** 2).item()
    lr_probe_summary[probe_lr] = (round(probe_value.item(), 6), round(final_probe_loss, 6))
print("lr → (final value, final loss):", lr_probe_summary)
print("가장 큰 lr이 더 나쁨:", lr_probe_summary[1.1][1] > lr_probe_summary[0.1][1])

### 관찰

이 고정된 1차원 문제와 12번 update에서는 `lr=1.1`이 목표를 지나쳐 오가며 loss가 커졌음. 이것은 “큰 learning rate가 언제나 더 빠르다”는 말을 반박하는 한 가지 증거일 뿐, 모든 문제의 최적 learning rate를 정해 주는 규칙은 아님.

In [ ]:
comparison_x = torch.linspace(-1, 1, 41).reshape(-1, 1)
comparison_y = 2 * comparison_x - 0.5
class ComparisonLine(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)
    def forward(self, value):
        return self.linear(value)

curve_base = ComparisonLine()
with torch.no_grad():
    curve_base.linear.weight.fill_(-0.75)
    curve_base.linear.bias.fill_(0.75)
curve_base_state = copy.deepcopy(curve_base.state_dict())
curve_conditions = [
    ("SGD lr=.01", torch.optim.SGD, 0.01),
    ("SGD lr=.10", torch.optim.SGD, 0.10),
    ("Adam lr=.01", torch.optim.Adam, 0.01),
    ("Adam lr=.10", torch.optim.Adam, 0.10),
]
curve_summary = {}
for label, optimizer_class, learning_rate in curve_conditions:
    model = ComparisonLine()
    model.load_state_dict(curve_base_state)
    optimizer = optimizer_class(model.parameters(), lr=learning_rate)
    checkpoints = []
    for step in range(200):
        optimizer.zero_grad()
        prediction = model(comparison_x)
        loss = ((prediction - comparison_y) ** 2).mean()
        loss.backward()
        optimizer.step()
        if step in (0, 9, 49, 99, 199):
            checkpoints.append(round(loss.item(), 6))
    curve_summary[label] = checkpoints
for label, checkpoints in curve_summary.items():
    print(label, "MSE@1/10/50/100/200=", checkpoints)

![SGD와 Adam 비교를 읽는 기준](assets/day02/sgd_adam_no_universal_winner_dark.svg)

이 비교는 같은 작은 regression data, model structure, 정확한 initialization, 200-step budget을 고정하고 optimizer와 learning rate 조건을 명시했음. 같은 lr `.10`의 이 실행에서는 SGD loss가 여러 checkpoint에서 더 낮아 “Adam이 항상 이긴다”를 반박함. 반대로 이 한 사례로 SGD의 보편적 우승도 주장할 수 없음.

Adam을 처음 읽는 데 필요한 직관만 남김. 최근 gradient 방향을 이동 평균처럼 모아 흔들림을 완화하는 부분은 momentum과 닮았고, gradient 제곱의 이동 평균을 이용해 parameter별 update 크기를 조절하는 부분은 RMSProp과 닮았음. 정확히는 Adam이 gradient와 gradient 제곱의 running average를 함께 관리함.

이 설명은 optimizer를 알아보는 경계임. 수식 유도나 “Adam이 항상 더 좋음”을 뜻하지 않으며, 실제 선택은 data·model·learning rate·관찰한 curve에 근거함.

### Practice PC-06 · PC-07

이번 Practice에서 확인할 것: PC-06은 canonical loop와 gradient lifecycle의 책임을 구분함.
이번 Practice에서 확인할 것: PC-07은 같은 조건의 curve 근거와 Adam의 두 running-average 직관을 읽음.

- PC-06: canonical loop와 gradient lifecycle
- PC-07: 네 200-step curve의 조건·근거와 Adam의 두 running-average 직관을 읽기

답을 확인한 뒤 **Dataset에서 현재 batch까지** 제목으로 돌아옴.

## Dataset에서 현재 batch까지

500개 전체를 한 번에 직접 잘라 전달하는 대신, “어떤 sample을 주는가”와 “어떤 순서와 크기의 묶음으로 주는가”를 분리함.

- `TensorDataset`: 같은 index의 input과 target을 한 example로 묶음
- `DataLoader`: Dataset에서 batch를 만들어 반복 전달함

In [ ]:
day02_dataset = TensorDataset(day02_features, day02_targets)
day02_loader = DataLoader(day02_dataset, batch_size=64, shuffle=False, drop_last=False)
batch_sizes = []
for current_inputs, current_targets in day02_loader:
    batch_sizes.append(len(current_inputs))
    assert len(current_inputs) == len(current_targets)
print("dataset length:", len(day02_dataset))
print("loader length:", len(day02_loader))
print("batch sizes:", batch_sizes)
print("last batch:", batch_sizes[-1], "sum:", sum(batch_sizes))

![500개를 64개씩 전달하는 DataLoader](assets/day02/dataset_dataloader_batch_dark.svg)

`500 = 64×7 + 52`이므로 `drop_last=False`에서 8개 batch, 마지막은 52개임. 각 loop에서 prediction은 **현재 `current_targets`**와 비교해야 하며 전체 target을 잘못 가져오지 않음.

### 같은 계산 장소 규칙

같은 forward/loss 계산에 참여하는 model, input, target은 같은 device에 둠. GPU 이름을 외우는 것보다 이 불변 규칙이 먼저임.

In [ ]:
if torch.cuda.is_available():
    day02_device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    day02_device = torch.device("mps")
else:
    day02_device = torch.device("cpu")

device_model = ClassificationMLP().to(day02_device)
first_inputs, first_targets = next(iter(day02_loader))
first_inputs = first_inputs.to(day02_device)
first_targets = first_targets.to(day02_device)
first_logits = device_model(first_inputs)
print("selected device:", day02_device)
print("model / inputs / targets:", next(device_model.parameters()).device, first_inputs.device, first_targets.device)
print("logits / target shapes:", tuple(first_logits.shape), tuple(first_targets.shape))

![같은 계산에 참여하는 값의 device 규칙](assets/day02/device_same_place_dark.svg)

현재 실행은 사용 가능한 device를 선택함. 이 자료는 실제 GPU/MPS mismatch 오류를 재현했다고 주장하지 않으며, model만 옮기거나 input만 옮기면 충분하다고 말하지 않음.

### Practice PC-08

이번 Practice에서 확인할 것: Dataset·DataLoader·현재 batch·same-device 책임을 연결함.

Practice Q의 **PC-08**로 이동함.  
답을 확인한 뒤 **평가와 전체 pipeline** 제목으로 돌아옴.

## 평가와 전체 pipeline

평가에서는 두 질문을 따로 관리함.

1. 모델의 동작 mode는 학습용인가 평가용인가?
2. gradient 계산 기록이 필요한가?

`model.eval()`과 `torch.no_grad()`는 서로 다른 스위치임.

In [ ]:
mode_probe = nn.Sequential(nn.Linear(2, 2, bias=False), nn.Dropout(p=0.5))
with torch.no_grad():
    mode_probe[0].weight.copy_(torch.eye(2))
mode_input = torch.ones(1, 2, requires_grad=True)
torch.manual_seed(20260904)
mode_probe.train()
train_output = mode_probe(mode_input)
train_state = (mode_probe.training, torch.is_grad_enabled(), train_output.requires_grad)
mode_probe.eval()
eval_output = mode_probe(mode_input)
eval_state = (mode_probe.training, torch.is_grad_enabled(), eval_output.requires_grad)
with torch.no_grad():
    eval_no_grad_output = mode_probe(mode_input)
    eval_no_grad_state = (mode_probe.training, torch.is_grad_enabled(), eval_no_grad_output.requires_grad)
print("(model.training, grad enabled, output requires_grad)")
print("train / normal:", train_state)
print("eval  / normal:", eval_state)
print("eval  / no_grad:", eval_no_grad_state)
print("Dropout train output:", train_output.detach().tolist())
print("Dropout eval output:", eval_output.detach().tolist())

![모델 mode와 gradient 기록은 다른 스위치](assets/day02/train_eval_nograd_dark.svg)

`eval()`만으로 autograd가 꺼지지 않았고, `no_grad()`는 model mode를 바꾸지 않았음. 평가 뒤 학습을 다시 시작할 때는 `model.train()`으로 복귀함.

여기서는 차이를 눈으로 보기 위해 **학습 중에 중간 값 일부를 임의로 가리는 Dropout**을 작은 probe에만 넣었음. train에서는 일부 값이 바뀌고 eval에서는 모두 사용되는 출력 차이를 확인함. 앞의 `ClassificationMLP`에는 Dropout이 없으므로 그 모델에서는 mode 전환의 값 차이가 당장 보이지 않지만, 두 스위치의 책임은 여전히 구분함.

### 전체 pipeline을 역할로 연결하기

`Dataset → DataLoader → current batch → device → model/raw logits → loss → backward → optimizer`가 학습 흐름임. 평가는 `eval → no_grad → logits → argmax → correct/total`로 읽음.

아래 코드는 새 API를 한꺼번에 배우는 목록이 아님. 이미 배운 조각을 하나의 실행 가능한 흐름으로 다시 연결함.

In [ ]:
generator = torch.Generator().manual_seed(20260903)
training_loader = DataLoader(day02_dataset, batch_size=64, shuffle=True, generator=generator)
torch.manual_seed(27182)
pipeline_model = ClassificationMLP().to(day02_device)
pipeline_criterion = nn.CrossEntropyLoss()
pipeline_optimizer = torch.optim.Adam(pipeline_model.parameters(), lr=0.02)

for epoch in range(60):
    pipeline_model.train()
    for batch_inputs, batch_targets in training_loader:
        batch_inputs = batch_inputs.to(day02_device)
        batch_targets = batch_targets.to(day02_device)
        assert batch_inputs.ndim == 2 and batch_inputs.shape[1] == 2
        assert batch_targets.ndim == 1 and batch_targets.dtype == torch.int64
        assert next(pipeline_model.parameters()).device == batch_inputs.device == batch_targets.device
        pipeline_optimizer.zero_grad()
        batch_logits = pipeline_model(batch_inputs)
        assert batch_logits.shape == (len(batch_targets), 2)
        batch_loss = pipeline_criterion(batch_logits, batch_targets)
        batch_loss.backward()
        pipeline_optimizer.step()

pipeline_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_inputs, batch_targets in day02_loader:
        batch_inputs = batch_inputs.to(day02_device)
        batch_targets = batch_targets.to(day02_device)
        batch_logits = pipeline_model(batch_inputs)
        batch_predictions = batch_logits.argmax(dim=1)
        correct += (batch_predictions == batch_targets).sum().item()
        total += len(batch_targets)
pipeline_accuracy = correct / total
print("evaluation correct / total:", correct, "/", total)
print("evaluation accuracy:", round(pipeline_accuracy, 4))
print("model.training after eval:", pipeline_model.training)

![Day02 전체 pipeline 역할 지도](assets/day02/full_pipeline_day02_dark.svg)

Accuracy는 batch percentage를 단순 평균내지 않고, batch마다 맞힌 개수와 sample 수를 누적한 뒤 `correct / total`로 계산함. 마지막 batch가 52여도 denominator가 정확히 500이 됨.

### 실행됐지만 짝이 잘못될 수 있는 shape

Prediction `[n,1]`과 target `[n]`을 그대로 빼면 각 sample끼리의 네 쌍이 아니라 `[n,n]` 조합으로 늘어날 수 있음. scalar loss가 나왔다는 사실만으로 shape 계약이 맞았다고 결론 내리지 않음.

In [ ]:
broadcast_predictions = torch.tensor([[0.1], [0.9], [0.8], [0.2]])
flat_targets = torch.tensor([0.0, 1.0, 1.0, 0.0])
broadcast_difference = broadcast_predictions - flat_targets
correct_targets = flat_targets.reshape(-1, 1)
correct_difference = broadcast_predictions - correct_targets
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter("always")
    broadcast_loss = nn.functional.mse_loss(broadcast_predictions, flat_targets)
correct_loss = nn.functional.mse_loss(broadcast_predictions, correct_targets)
print("wrong pairwise shape:", tuple(broadcast_difference.shape), "scalar loss:", round(broadcast_loss.item(), 6))
print("correct pairwise shape:", tuple(correct_difference.shape), "scalar loss:", round(correct_loss.item(), 6))
print("warning captured:", len(captured) > 0)

### 통합 경계의 짧은 guard

- input: `[batch, 2]`, floating dtype
- target: `[batch]`, `torch.int64`
- logits: `[batch, 2]`, raw scores
- model/input/target: same device
- accuracy: current batch target과 비교, correct/total 누적

Guard는 학습을 대신하지 않지만, 실행 가능한 잘못된 연결을 더 일찍 드러냄.

### Practice PC-09

이번 Practice에서 확인할 것: `eval()`·`no_grad()`·accuracy·shape guard의 책임을 구분함.

Practice Q의 **PC-09**로 이동함.  
답을 확인한 뒤 **진단하고 다음 구조로 연결** 제목으로 돌아옴.

## 진단하고 다음 구조로 연결

진단 순서: **기대 계약 → 실제 관찰 → 불일치 → 최소 수정 → 재실행**. 에러가 나지 않은 코드도 의미가 잘못될 수 있음.

In [ ]:
guard_model = nn.Linear(2, 1)
wrong_dtype_input = torch.ones(1, 2, dtype=torch.float64)
try:
    guard_model(wrong_dtype_input)
except (RuntimeError, TypeError) as error:
    print("guarded real failure:", type(error).__name__)
    print("message:", str(error).splitlines()[0])

fixed_dtype_input = wrong_dtype_input.to(dtype=guard_model.weight.dtype)
fixed_output = guard_model(fixed_dtype_input)
print("repaired input/output:", fixed_dtype_input.dtype, tuple(fixed_output.shape))

이 실패는 현재 CPU에서 실제로 발생시켜 잡은 dtype 오류임. GPU/MPS device 오류를 재현한 것으로 바꾸어 말하지 않음. 또한 다음은 예외가 없어도 진단 대상임.

- Softmax를 CE 전에 넣어 현재 계약을 바꿈
- `step()` 뒤 gradient가 지워졌다고 가정함
- `[n,1]`과 `[n]` broadcasting을 짝별 비교로 오해함
- current batch prediction을 전체 target과 비교함

### Sprint Mission readiness · 답 대신 역할과 실행 계약 찾기

아래 낯선 함수에서 먼저 찾을 것

1. current batch와 device 이동
2. raw logits와 integer target
3. gradient clear / forward / loss / backward / update
4. 평가 mode / no_grad / argmax / correct-total

마지막에는 정답 조각이 아니라 notebook 실행 계약을 점검함.

- 필요한 import가 위에 있으며 새 kernel에서도 접근 가능한가?
- random 결과를 비교한다면 seed를 셀 안에서 고정했는가?
- 아래 셀이 수동 실행한 숨은 변수에 기대지 않는가?
- 저장 전 **Restart Kernel → Run All**로 위에서 아래까지 다시 확인했는가?

In [ ]:
def inspect_one_training_step(model, optimizer, criterion, inputs, targets, device):
    model.train()
    inputs = inputs.to(device)
    targets = targets.to(device)
    optimizer.zero_grad()
    logits = model(inputs)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer.step()
    return {"input": tuple(inputs.shape), "logits": tuple(logits.shape), "target": tuple(targets.shape)}

readiness_model = ClassificationMLP().to(day02_device)
readiness_optimizer = torch.optim.SGD(readiness_model.parameters(), lr=0.1)
readiness_inputs, readiness_targets = next(iter(day02_loader))
print(inspect_one_training_step(
    readiness_model, readiness_optimizer, nn.CrossEntropyLoss(),
    readiness_inputs, readiness_targets, day02_device,
))

### 다음 질문 1 · 이미지의 위치 관계

![이미지의 공간 구조를 묻는 CNN bridge](assets/day02/cnn_bridge_spatial_structure_dark.svg)

MLP는 feature를 섞을 수 있지만, 이미지에서 가까운 pixel끼리의 관계를 특별히 사용하도록 설계한 것은 아님. **공간 구조를 계산에 직접 쓰려면 어떤 구조가 필요할까?**라는 질문만 남기며 CNN 전체 내용은 여기서 전개하지 않음.

### 다음 질문 2 · 순서와 이전 정보

![순서와 상태를 묻는 RNN bridge](assets/day02/rnn_bridge_sequence_state_dark.svg)

순서가 바뀌면 의미가 바뀌는 데이터에서는 앞에서 본 정보를 다음 시점에 넘길 구조가 필요할 수 있음. **이전 정보를 다음 계산으로 어떻게 전달할까?**라는 질문만 남기며 RNN/BPTT는 여기서 가르치지 않음.

### 마지막 회수

낯선 PyTorch 코드에서도 다음 역할을 순서대로 찾음.

`data → current batch → device → model/parameter → prediction(logits) → loss → gradient → update → evaluation`

각 단계에서 값 하나만 보지 않고 shape, dtype, device, 현재 batch의 짝을 함께 확인함. 마지막으로 import·seed·위에서 아래 실행 순서를 점검해 숨은 상태를 제거함.

### Practice PC-10 · PC-11

이번 Practice에서 확인할 것: PC-10은 계약을 기준으로 오류와 다음 구조의 경계를 진단함.
이번 Practice에서 확인할 것: PC-11은 Day1 연산 선택·짧은 API 확인·clean Run All을 회수함.

- PC-10: 계약을 기준으로 오류와 다음 구조를 진단함
- PC-11: Day1 연산 선택, dtype·Sigmoid·MSE API 관계, clean Run All을 한 번에 회수함

PC-11은 기존 진단 장면을 비대하게 만들지 않기 위한 분리이며, PC-01~10의 번호나 역할을 바꾸지 않음. 답 확인은 별도 Practice A의 같은 PC 번호에서 수행함.

## 오늘 마무리

낯선 코드에서도 역할과 계약을 순서대로 찾음. 마지막으로 “실행됐는가?”뿐 아니라 “의도한 shape·dtype·device·현재 batch의 짝인가?”를 함께 확인함.